# Day 19 - Bronze Ingestion Pattern

**Topic:** Bronze Ingestion Pattern  
**Main environment:** Microsoft Fabric Notebook + Fabric Lakehouse  
**Focus:** ฝึกออกแบบ Bronze ingestion ที่เก็บ raw data พร้อม audit columns เช่น `ingestion_timestamp`, `source_file`, `batch_id` และ `run_id`

วันนี้เราจะเริ่ม Phase 4 ด้วย Bronze layer ซึ่งเป็นจุดรับข้อมูลดิบเข้าสู่ Lakehouse โดยโฟกัสที่ traceability ก่อน cleansing จริงใน Silver เป้าหมายคืออ่าน raw file, preserve ค่าเดิมให้มากที่สุด, เพิ่ม metadata ที่ช่วยตามรอยได้ และเขียนเป็น Bronze Delta table ใน Fabric Lakehouse


## Goal of the day

1. อธิบายได้ว่า Bronze layer ควรเก็บอะไรและไม่ควรรีบแก้อะไร
2. อ่าน raw CSV source ด้วย explicit schema แบบเน้น raw preservation ได้
3. เพิ่ม audit columns ที่สำคัญ เช่น `ingestion_timestamp`, `source_file`, `source_system`, `batch_id`, `run_id`
4. เขียนข้อมูลเข้า Bronze table แบบ append-only pattern ได้
5. สร้าง simple ingestion log เพื่อเช็กจำนวน records และ source traceability เบื้องต้นได้


## Why it matters in production

ใน production pipeline Bronze ingestion สำคัญมาก เพราะเป็นจุดแรกที่ข้อมูลจาก source เข้ามาอยู่ใน Lakehouse ถ้าชั้นนี้ไม่มี traceability เวลาข้อมูลผิด เราจะตอบคำถามพื้นฐานได้ยาก เช่น record นี้มาจากไฟล์ไหน, run ไหน, batch ไหน, ingest ตอนเวลาใด และ source ส่งค่าดิบมาอย่างไร

Bronze layer ที่ดีช่วยเรื่อง:

- reprocessing เมื่อ logic ใน Silver/Gold เปลี่ยน
- investigation เมื่อ source ส่งข้อมูลผิดหรือ format เปลี่ยน
- auditability ว่าข้อมูลถูก ingest เข้ามาเมื่อไรและจาก source ไหน
- debugging เมื่อ pipeline run สำเร็จแต่ downstream ได้ตัวเลขผิด
- ลดความเสี่ยงจากการ clean/drop ข้อมูลเร็วเกินไปตั้งแต่แรก

Production takeaway:

> Bronze ไม่ใช่ที่สำหรับทำข้อมูลให้สวยที่สุด แต่เป็นที่สำหรับเก็บหลักฐานดิบให้ตามรอยและนำไป process ต่อได้อย่างมั่นใจ


## Key concepts

| Concept | Meaning |
|---|---|
| Bronze layer | ชั้นแรกของ Lakehouse ที่รับ raw data จาก source เข้ามาเก็บเป็น Bronze table ที่ query ต่อได้ |
| Raw preservation | เก็บค่าจาก source ให้ใกล้เคียงต้นฉบับมากที่สุดก่อนทำ cleansing หนัก ๆ |
| Source traceability | ความสามารถในการตามกลับได้ว่า record มาจาก source file, source system, batch หรือ run ใด |
| Audit columns | Technical columns ที่เพิ่มตอน ingest เช่น `ingestion_timestamp`, `source_file`, `batch_id`, `run_id` |
| Append-only mindset | Bronze มักถูกเขียนแบบเพิ่มข้อมูลใหม่เข้าไป ไม่ overwrite ทิ้งง่าย ๆ เพื่อรักษาหลักฐานการรับข้อมูล |
| Batch ID | รหัสของชุดข้อมูลที่ ingest เช่น batch ประจำวันที่ หรือ file drop รอบหนึ่ง |
| Run ID | รหัสของ pipeline/notebook run ใช้แยกการ execute แต่ละครั้ง |

## Concept explanation

### Bronze layer คืออะไร?

Bronze layer คือ landing zone ในรูปแบบที่ query ได้ โดยปกติจะเก็บข้อมูลที่มาจาก source ให้ใกล้ raw ที่สุด แต่เพิ่ม technical metadata เพื่อให้ตรวจสอบย้อนหลังได้

ตัวอย่างคำถามที่ Bronze ควรช่วยตอบได้:

- record นี้มาจาก source file ไหน?
- ingest เข้ามาเมื่อไร?
- อยู่ใน batch ไหน?
- เกิดจาก pipeline run ไหน?
- source ส่งค่า raw มาอย่างไร ก่อนที่ Silver จะ cast/clean/standardize?

> Bronze = raw evidence + technical traceability

### ทำไมไม่ clean หนัก ๆ ใน Bronze?

ถ้าเรา cast, drop, deduplicate หรือแก้ค่าเร็วเกินไปตั้งแต่ Bronze เราอาจเสียหลักฐานสำคัญจาก source เช่น `amount_text = "abc"` หรือ `event_time_text = "not-a-timestamp"`

ใน notebook วันนี้เราจะเก็บ column ดิบเป็น string ก่อน แล้วค่อยให้ Day 20 Silver Cleansing เป็นจุดที่ cast type, standardize, deduplicate และแยก valid/invalid logic ต่อไป

### Audit columns ที่ควรมีใน Bronze

ชุดเล็ก ๆ ที่ควรเริ่มจำให้ได้:

```text
ingestion_timestamp = เวลาที่ record ถูก ingest เข้า Lakehouse
source_file         = path หรือชื่อไฟล์ต้นทาง
source_system       = ระบบต้นทาง เช่น CRM, ERP, API
batch_id            = ชุดข้อมูลที่ ingest
run_id              = execution run ของ notebook/pipeline
```

### Bronze write mode

สำหรับ learning lab วันนี้เราจะใช้ `append` เพื่อจำลอง pattern แบบรับ batch ใหม่เข้ามาเรื่อย ๆ

ใน production จริง ห้ามใช้ `overwrite` กับ Bronze table โดยไม่ตั้งใจ เพราะอาจลบหลักฐานดิบของ batch ก่อนหน้าออกไปทั้งหมด


## Mermaid diagram: Bronze Ingestion Flow

```mermaid
flowchart LR
    A[Raw source file] --> B[Read with explicit all-string schema]
    B --> C[Preserve raw columns]
    C --> D[Add audit columns]
    D --> E[Write append to Bronze Delta table]
    E --> F[Read Bronze for validation]
    E --> G[Write simple ingestion log]

    D -. traceability .-> H[source_file / source_system / batch_id / run_id]
```

Key idea:

> Bronze ingestion ควรทำให้ข้อมูลดิบกลายเป็น table ที่ตามรอยได้ ไม่ใช่รีบเปลี่ยนทุกอย่างให้ clean ตั้งแต่ step แรก


## Setup / imports


In [1]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

import os      # Work with OS paths and file system.
import shutil  # Copy, move, or remove files/folders.
from datetime import datetime

StatementMeta(, e224dccf-d9ed-4f86-bb8b-64f96f8f0c7f, 3, Finished, Available, Finished, False)

## Create mock data

Notebook นี้จะสร้าง raw CSV files เล็ก ๆ ไว้ใน Fabric Lakehouse Files เพื่อจำลอง source file drop

Path ที่ใช้:

```text
/lakehouse/default/Files/day19_bronze_ingestion/incoming/customer_events/
```


In [33]:
# Fabric has two useful path styles:
# - Mount path: use with Python file APIs such as open(), os, and shutil.
# - Spark path: use with spark.read / spark.write.
base_path_mount = "/lakehouse/default/Files/day19_bronze_ingestion"
base_path_spark = "Files/day19_bronze_ingestion"

incoming_path_mount = f"{base_path_mount}/incoming/customer_events"
incoming_path_spark = f"{base_path_spark}/incoming/customer_events"

bronze_table_name = "day19_bronze_customer_events"
bronze_log_table_name = "day19_bronze_ingestion_log"

if os.path.exists(base_path_mount):
    shutil.rmtree(base_path_mount)

os.makedirs(incoming_path_mount, exist_ok=True)

spark.sql(f"DROP TABLE IF EXISTS {bronze_table_name}")
spark.sql(f"DROP TABLE IF EXISTS {bronze_log_table_name}")

batch_001_csv = """event_id,customer_id,customer_name,city,event_time_text,amount_text,payment_status
E001,C001,Alice,Bangkok,2026-06-01 08:01:00,1200.50,paid
E002,C002,Bob,Chiang Mai,2026-06-01 09:15:00,850.00,paid
E003,C003,Charlie,Phuket,not-a-timestamp,abc,failed
E004,,Diana,Bangkok,2026-06-01 10:30:00,430.00,pending
"""

batch_002_csv = """event_id,customer_id,customer_name,city,event_time_text,amount_text,payment_status
E005,C004,Ethan,Rayong,2026-06-02 08:45:00,990.00,paid
E006,C002,Bob,Chiang Mai,2026-06-02 11:20:00,300.00,paid
E007,C005,Fah,Nakhon Pathom,2026-06-02 12:05:00,-50.00,refunded
"""

# Use mount paths when creating files with Python.
batch_001_file_path = f"{incoming_path_mount}/customer_events_batch_001.csv"
batch_002_file_path = f"{incoming_path_mount}/customer_events_batch_002.csv"

# Use Spark paths when reading files with Spark.
batch_001_spark_path = f"{incoming_path_spark}/customer_events_batch_001.csv"
batch_002_spark_path = f"{incoming_path_spark}/customer_events_batch_002.csv"

with open(batch_001_file_path, "w", encoding="utf-8") as file:
    file.write(batch_001_csv)

with open(batch_002_file_path, "w", encoding="utf-8") as file:
    file.write(batch_002_csv)

print("Created source files:")
print(batch_001_file_path)
print(batch_002_file_path)

print("Spark read paths:")
print(batch_001_spark_path)
print(batch_002_spark_path)

StatementMeta(, e224dccf-d9ed-4f86-bb8b-64f96f8f0c7f, 35, Finished, Available, Finished, False)

Created source files:
/lakehouse/default/Files/day19_bronze_ingestion/incoming/customer_events/customer_events_batch_001.csv
/lakehouse/default/Files/day19_bronze_ingestion/incoming/customer_events/customer_events_batch_002.csv
Spark read paths:
Files/day19_bronze_ingestion/incoming/customer_events/customer_events_batch_001.csv
Files/day19_bronze_ingestion/incoming/customer_events/customer_events_batch_002.csv


## PySpark code examples

ใน section นี้เราจะสร้าง Bronze ingestion pattern แบบเล็ก ๆ แต่ใกล้กับงานจริง:

1. กำหนด raw schema
2. อ่าน source file
3. เพิ่ม audit columns
4. เขียน Bronze table
5. สร้าง ingestion log
6. append batch ใหม่เข้า Bronze table เดิม


### Example 1: Define raw schema

สำหรับ Bronze ingestion วันนี้ เราจะอ่านทุก source column เป็น `StringType()` ก่อน เพื่อ preserve raw value จาก source ให้มากที่สุด

เหตุผลคือ source อาจส่งข้อมูลผิด format เช่น date เป็น `not-a-timestamp` หรือ amount เป็น `abc` ถ้า cast ตั้งแต่ Bronze อาจทำให้เสียหลักฐานที่จำเป็นสำหรับ investigation


In [34]:
raw_schema = T.StructType([
    T.StructField("event_id", T.StringType(), True),
    T.StructField("customer_id", T.StringType(), True),
    T.StructField("customer_name", T.StringType(), True),
    T.StructField("city", T.StringType(), True),
    T.StructField("event_time_text", T.StringType(), True),
    T.StructField("amount_text", T.StringType(), True),
    T.StructField("payment_status", T.StringType(), True),
])

raw_columns = [field.name for field in raw_schema.fields]

print(raw_columns)

StatementMeta(, e224dccf-d9ed-4f86-bb8b-64f96f8f0c7f, 36, Finished, Available, Finished, False)

['event_id', 'customer_id', 'customer_name', 'city', 'event_time_text', 'amount_text', 'payment_status']


### Example 2: Read first source file

อ่าน batch แรกจาก raw CSV file ด้วย explicit schema ที่กำหนดเอง

จุดที่ต้องสังเกต:

- `event_time_text` ยังเป็น string
- `amount_text` ยังเป็น string
- invalid raw value ยังถูกเก็บไว้ ไม่ถูก drop เงียบ ๆ


In [35]:
df_raw_batch_001 = (
    spark.read
    .option("header", True)
    .schema(raw_schema)
    .csv(batch_001_spark_path)
)

df_raw_batch_001.show(truncate=False)
df_raw_batch_001.printSchema()

source_record_count_001 = df_raw_batch_001.count()
print("Raw batch 001 count:", source_record_count_001)

StatementMeta(, e224dccf-d9ed-4f86-bb8b-64f96f8f0c7f, 37, Finished, Available, Finished, False)

+--------+-----------+-------------+----------+-------------------+-----------+--------------+
|event_id|customer_id|customer_name|city      |event_time_text    |amount_text|payment_status|
+--------+-----------+-------------+----------+-------------------+-----------+--------------+
|E001    |C001       |Alice        |Bangkok   |2026-06-01 08:01:00|1200.50    |paid          |
|E002    |C002       |Bob          |Chiang Mai|2026-06-01 09:15:00|850.00     |paid          |
|E003    |C003       |Charlie      |Phuket    |not-a-timestamp    |abc        |failed        |
|E004    |NULL       |Diana        |Bangkok   |2026-06-01 10:30:00|430.00     |pending       |
+--------+-----------+-------------+----------+-------------------+-----------+--------------+

root
 |-- event_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- event_time_text: string (nullable = true)
 |-- amount_text: strin

### Example 3: Add Bronze audit columns

ขั้นตอนนี้คือหัวใจของ Bronze ingestion pattern

เราจะเพิ่ม columns ที่ไม่ได้มาจาก source โดยตรง แต่ช่วยให้ trace กลับได้:

- `ingestion_timestamp`
- `ingestion_date`
- `source_file`
- `source_system`
- `batch_id`
- `run_id`


In [36]:
def add_bronze_audit_columns(df_raw, source_system: str, batch_id: str, run_id: str):

    return (
        df_raw
        .withColumn("ingestion_timestamp", F.current_timestamp())
        .withColumn("ingestion_date", F.to_date(F.col("ingestion_timestamp")))
        .withColumn("source_file", F.input_file_name())  # input_file_name() returns the source file path.
        .withColumn("source_system", F.lit(source_system))
        .withColumn("batch_id", F.lit(batch_id))
        .withColumn("run_id", F.lit(run_id))
    )

run_id_001 = "run_20260603_001"
batch_id_001 = "batch_20260601_customer_events"

df_bronze_batch_001 = add_bronze_audit_columns(
    df_raw=df_raw_batch_001,
    source_system="crm_mock",
    batch_id=batch_id_001,
    run_id=run_id_001,
)

df_bronze_batch_001.show(truncate=False)
df_bronze_batch_001.printSchema()

StatementMeta(, e224dccf-d9ed-4f86-bb8b-64f96f8f0c7f, 38, Finished, Available, Finished, False)

+--------+-----------+-------------+----------+-------------------+-----------+--------------+--------------------------+--------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------+------------------------------+----------------+
|event_id|customer_id|customer_name|city      |event_time_text    |amount_text|payment_status|ingestion_timestamp       |ingestion_date|source_file                                                                                                                                                                                                                             |source_system|batch_id                      |run_id          |
+--------+-----------+-------------+----------+-------------------+-----------+--------------+--------------------------+--------------+

### Example 4: Write first batch to Bronze table

เราจะเขียน Bronze table ด้วย `append` mode ถึงแม้ตอนนี้จะเป็น batch แรกก็ตาม เพื่อจำ pattern ว่า Bronze มักรับข้อมูลใหม่เข้ามาเรื่อย ๆ


In [37]:
(
    df_bronze_batch_001.write
    .format("delta")
    .mode("append")
    .saveAsTable(bronze_table_name)
)

print(f"Written first batch to table: {bronze_table_name}")

StatementMeta(, e224dccf-d9ed-4f86-bb8b-64f96f8f0c7f, 39, Finished, Available, Finished, False)

Written first batch to table: day19_bronze_customer_events


### Example 5: Create simple ingestion log

Bronze table เก็บ record-level data ส่วน ingestion log เก็บ run-level summary แบบเบื้องต้น

วันนี้เราจะทำ simple log เพื่อให้เห็น pattern ก่อน ยังไม่ทำ control/audit framework เต็มรูปแบบ


In [43]:
def build_ingestion_log(
    df_bronze,
    target_table_name: str,
    source_system: str,
    batch_id: str,
    run_id: str,
    source_record_count: int,
):

    return (
        df_bronze
        .agg(
            F.count("*").alias("target_record_count"),
            F.countDistinct("source_file").alias("source_file_count"),
        )
        .withColumn("target_table", F.lit(target_table_name))  # lit() adds a fixed value from a Python variable.
        .withColumn("source_system", F.lit(source_system))
        .withColumn("batch_id", F.lit(batch_id))
        .withColumn("run_id", F.lit(run_id))
        .withColumn("source_record_count", F.lit(source_record_count))
        .withColumn("log_created_at", F.current_timestamp())
        .select(
            "target_table",
            "source_system",
            "batch_id",
            "run_id",
            "source_record_count",
            "target_record_count",
            "source_file_count",
            "log_created_at",
        )
    )

# Read committed Bronze rows before building the ingestion log.
df_bronze_committed_batch_001 = (
    spark.read.table(bronze_table_name)
    .filter(
        (F.col("batch_id") == batch_id_001) &
        (F.col("run_id") == run_id_001)
    )
)

log_batch_001 = build_ingestion_log(
    df_bronze=df_bronze_committed_batch_001,
    target_table_name=bronze_table_name,
    source_system="crm_mock",
    batch_id=batch_id_001,
    run_id=run_id_001,
    source_record_count=source_record_count_001,
)

(
    log_batch_001.write
    .format("delta")
    .mode("append")
    .saveAsTable(bronze_log_table_name)
)

# Read the committed log row from the log table.
df_committed_log_batch_001 = (
    spark.read.table(bronze_log_table_name)
    .filter(
        (F.col("batch_id") == batch_id_001) &
        (F.col("run_id") == run_id_001)
    )
)

df_committed_log_batch_001.show(truncate=False)

StatementMeta(, e224dccf-d9ed-4f86-bb8b-64f96f8f0c7f, 45, Finished, Available, Finished, False)

+----------------------------+-------------+------------------------------+----------------+-------------------+-------------------+-----------------+--------------------------+
|target_table                |source_system|batch_id                      |run_id          |source_record_count|target_record_count|source_file_count|log_created_at            |
+----------------------------+-------------+------------------------------+----------------+-------------------+-------------------+-----------------+--------------------------+
|day19_bronze_customer_events|crm_mock     |batch_20260601_customer_events|run_20260603_001|4                  |4                  |1                |2026-06-03 13:59:53.931465|
+----------------------------+-------------+------------------------------+----------------+-------------------+-------------------+-----------------+--------------------------+



### Example 6: Read Bronze table back and validate traceability

หลัง write แล้วต้องอ่านกลับมาตรวจอย่างน้อย 3 เรื่อง:

- row count ถูกต้องไหม
- audit columns มีครบไหม
- source traceability กลับไปถึง file / batch / run ได้ไหม


In [44]:
df_bronze_table = spark.read.table(bronze_table_name)

df_bronze_table.select(
    "event_id",
    "customer_id",
    "event_time_text",
    "amount_text",
    "ingestion_timestamp",
    "source_file",
    "batch_id",
    "run_id",
).show(truncate=False)

print("Bronze table count:", df_bronze_table.count())

StatementMeta(, e224dccf-d9ed-4f86-bb8b-64f96f8f0c7f, 46, Finished, Available, Finished, False)

+--------+-----------+-------------------+-----------+--------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------------------+----------------+
|event_id|customer_id|event_time_text    |amount_text|ingestion_timestamp       |source_file                                                                                                                                                                                                                             |batch_id                      |run_id          |
+--------+-----------+-------------------+-----------+--------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

### Example 7: Append second batch using a reusable ingestion function

ตอนนี้เราจะรวม logic เป็น function เล็ก ๆ เพื่อ ingest batch ใหม่เข้า Bronze table เดิม

Function นี้ยังเป็น learning lab version ไม่ใช่ framework เต็มรูปแบบ แต่ช่วยให้เห็น pattern ชัดขึ้น


In [45]:
def ingest_customer_event_batch(input_file_path: str, source_system: str, batch_id: str, run_id: str):
    # Read raw source with explicit all-string schema.
    df_raw = (
        spark.read
        .option("header", True)
        .schema(raw_schema)
        .csv(input_file_path)
    )

    source_record_count = df_raw.count()

    # Add traceability columns.
    df_bronze = add_bronze_audit_columns(
        df_raw=df_raw,
        source_system=source_system,
        batch_id=batch_id,
        run_id=run_id,
    )

    # Append to Bronze table.
    (
        df_bronze.write
        .format("delta")
        .mode("append")
        .saveAsTable(bronze_table_name)
    )

    # Read committed Bronze rows before building the ingestion log.
    df_bronze_committed = (
        spark.read.table(bronze_table_name)
        .filter(
            (F.col("batch_id") == batch_id) &
            (F.col("run_id") == run_id)
        )
    )

    # Build log from committed Bronze rows.
    log_df = build_ingestion_log(
        df_bronze=df_bronze_committed,
        target_table_name=bronze_table_name,
        source_system=source_system,
        batch_id=batch_id,
        run_id=run_id,
        source_record_count=source_record_count,
    )

    (
        log_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(bronze_log_table_name)
    )

    # Read committed log rows from the log table.
    log_committed = (
        spark.read.table(bronze_log_table_name)
        .filter(
            (F.col("batch_id") == batch_id) &
            (F.col("run_id") == run_id)
        )
    )

    return df_bronze_committed, log_committed

run_id_002 = "run_20260603_002"
batch_id_002 = "batch_20260602_customer_events"

# Tuple unpacking: a, b = c, d means a = c and b = d.
df_bronze_batch_002, log_batch_002 = ingest_customer_event_batch(
    input_file_path=batch_002_spark_path,
    source_system="crm_mock",
    batch_id=batch_id_002,
    run_id=run_id_002,
)

df_bronze_batch_002.show(truncate=False)
log_batch_002.show(truncate=False)

StatementMeta(, e224dccf-d9ed-4f86-bb8b-64f96f8f0c7f, 47, Finished, Available, Finished, False)

+--------+-----------+-------------+-------------+-------------------+-----------+--------------+--------------------------+--------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------+------------------------------+----------------+
|event_id|customer_id|customer_name|city         |event_time_text    |amount_text|payment_status|ingestion_timestamp       |ingestion_date|source_file                                                                                                                                                                                                                             |source_system|batch_id                      |run_id          |
+--------+-----------+-------------+-------------+-------------------+-----------+--------------+--------------------------+------

### Example 8: Check Bronze table and ingestion log after multiple batches

หลัง ingest หลาย batch แล้ว เราควรเห็นว่า Bronze table มี records จากหลาย `batch_id` และหลาย `run_id`


In [47]:
df_bronze_all = spark.read.table(bronze_table_name)
df_ingestion_log = spark.read.table(bronze_log_table_name)

print("Bronze total count:", df_bronze_all.count())

bronze_summary = (
    df_bronze_all
    .groupBy("source_system", "batch_id", "run_id")
    .agg(
        F.count("*").alias("target_record_count"),
        F.countDistinct("source_file").alias("source_file_count"),
    )
    .orderBy("batch_id", "run_id")
)

bronze_summary.show(truncate=False)
df_ingestion_log.orderBy("batch_id", "run_id").show(truncate=False)

StatementMeta(, e224dccf-d9ed-4f86-bb8b-64f96f8f0c7f, 49, Finished, Available, Finished, False)

Bronze total count: 7
+-------------+------------------------------+----------------+-------------------+-----------------+
|source_system|batch_id                      |run_id          |target_record_count|source_file_count|
+-------------+------------------------------+----------------+-------------------+-----------------+
|crm_mock     |batch_20260601_customer_events|run_20260603_001|4                  |1                |
|crm_mock     |batch_20260602_customer_events|run_20260603_002|3                  |1                |
+-------------+------------------------------+----------------+-------------------+-----------------+

+----------------------------+-------------+------------------------------+----------------+-------------------+-------------------+-----------------+--------------------------+
|target_table                |source_system|batch_id                      |run_id          |source_record_count|target_record_count|source_file_count|log_created_at            |
+--------

## Quick recap

| Question | Answer |
|---|---|
| Bronze layer เอาไว้ทำอะไร? | รับและเก็บ raw data พร้อม traceability ก่อนส่งต่อไป Silver |
| Bronze ควร clean ข้อมูลหนักไหม? | โดยทั่วไปไม่ควร clean หนัก เพราะต้อง preserve raw evidence ไว้ก่อน |
| Audit columns ที่สำคัญมีอะไรบ้าง? | `ingestion_timestamp`, `source_file`, `source_system`, `batch_id`, `run_id` |
| ทำไมต้องมี `source_file`? | เพื่อ trace กลับได้ว่า record มาจากไฟล์ไหนตอนเกิด issue |
| ทำไมใช้ `append` กับ Bronze? | เพื่อเก็บประวัติการรับข้อมูลแต่ละ batch ไม่ overwrite หลักฐานดิบทิ้งง่าย ๆ |
| Silver จะทำอะไรต่อจาก Bronze? | Cast type, clean, standardize, deduplicate และ apply DQ logic |


## Exercises

### Exercise 1: Inspect raw preservation in Bronze

ตรวจว่า invalid raw values ยังอยู่ใน Bronze table หรือไม่

Requirements:

- อ่าน `day19_bronze_customer_events`
- filter records ที่ `event_time_text = "not-a-timestamp"` หรือ `amount_text = "abc"`
- แสดง audit columns สำคัญด้วย

Expected result:

- ต้องเห็น record `E003`
- ค่า invalid raw ยังไม่ถูก drop หรือแก้หายไป


In [48]:
df_bronze_all = spark.read.table(bronze_table_name)

invalid_raw_records = (
    df_bronze_all
    .filter(
        (F.col("event_time_text") == "not-a-timestamp") |
        (F.col("amount_text") == "abc")
    )
    .select(
        "event_id",
        "customer_id",
        "event_time_text",
        "amount_text",
        "source_file",
        "batch_id",
        "run_id",
    )
)

invalid_raw_records.show(truncate=False)
print("Invalid raw record count:", invalid_raw_records.count())

StatementMeta(, e224dccf-d9ed-4f86-bb8b-64f96f8f0c7f, 50, Finished, Available, Finished, False)

+--------+-----------+---------------+-----------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------------------+----------------+
|event_id|customer_id|event_time_text|amount_text|source_file                                                                                                                                                                                                                             |batch_id                      |run_id          |
+--------+-----------+---------------+-----------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------------------+----------------+
|E00

### Exercise 2: Build a source-file level audit summary

สร้าง summary เพื่อตอบว่าแต่ละ source file ถูก commit เข้า Bronze table กี่ records

Requirements:

- group by `source_system`, `source_file`, `batch_id`, `run_id`
- count records ที่อยู่ใน Bronze table
- แสดงผลเรียงตาม `batch_id`, `run_id`

Expected result:

- batch 001 ควรมี `target_record_count = 4`
- batch 002 ควรมี `target_record_count = 3`


In [49]:
source_file_audit_summary = (
    df_bronze_all
    .groupBy("source_system", "batch_id", "run_id", "source_file")
    .agg(F.count("*").alias("target_record_count"))
    .orderBy("batch_id", "run_id", "source_file")
)

source_file_audit_summary.show(truncate=False)

StatementMeta(, e224dccf-d9ed-4f86-bb8b-64f96f8f0c7f, 51, Finished, Available, Finished, False)

+-------------+------------------------------+----------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------+
|source_system|batch_id                      |run_id          |source_file                                                                                                                                                                                                                             |target_record_count|
+-------------+------------------------------+----------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------+
|crm_mock     |batch_20260601_customer_events|run

### Exercise 3: Create a minimal Bronze quality visibility check

วันนี้ยังไม่ทำ DQ เต็มรูปแบบ แต่เราสามารถทำ visibility check เบื้องต้นได้ว่า raw records มี missing key กี่แถว

Requirements:

- สร้าง column `has_missing_customer_id`
- group by `batch_id` และนับจำนวน missing `customer_id`
- ไม่ drop records ออกจาก Bronze

Expected result:

- batch 001 มี `missing_customer_id_count = 1`
- ข้อมูลยังอยู่ใน Bronze table เหมือนเดิม


In [55]:
missing_key_visibility = (
    df_bronze_all
    .withColumn("has_missing_customer_id", F.col("customer_id").isNull() | (F.trim(F.col("customer_id")) == ""))
    .groupBy("batch_id")
    .agg(
        F.count("*").alias("total_records"),
        F.sum(F.col("has_missing_customer_id").cast("int")).alias("missing_customer_id_count"),  # Cast Boolean to int so true = 1 and false = 0 before summing.
    )
    .orderBy("batch_id")
)

missing_key_visibility.show(truncate=False)

StatementMeta(, e224dccf-d9ed-4f86-bb8b-64f96f8f0c7f, 57, Finished, Available, Finished, False)

+------------------------------+-------------+-------------------------+
|batch_id                      |total_records|missing_customer_id_count|
+------------------------------+-------------+-------------------------+
|batch_20260601_customer_events|4            |1                        |
|batch_20260602_customer_events|3            |0                        |
+------------------------------+-------------+-------------------------+



## Common mistakes

1. **Clean ข้อมูลหนักเกินไปตั้งแต่ Bronze**

   Bronze ควร preserve raw evidence ก่อน ถ้า cast/drop/deduplicate เร็วเกินไป อาจเสียข้อมูลสำหรับ investigation

2. **ลืม `source_file` หรือ `source_system`**

   ถ้า record มีปัญหา แต่ไม่รู้ว่ามาจากไฟล์หรือระบบไหน การ debug จะยากขึ้นมาก

3. **ใช้ `overwrite` กับ Bronze table โดยไม่ตั้งใจ**

   `overwrite` อาจลบ batch ก่อนหน้าทิ้งทั้งหมด สำหรับ Bronze ควรคิดแบบ append-first

4. **ไม่มี `batch_id` หรือ `run_id`**

   ถ้ามีหลายรอบการ ingest ในวันเดียวกัน การไม่มี run/batch identifier จะทำให้แยก incident หรือ rerun ได้ยาก

5. **เอา Bronze ingestion log ไปผูกกับ framework ใหญ่เร็วเกินไป**

   วันนี้ให้เข้าใจ pattern ก่อน แค่ simple log ก็พอ ส่วน audit framework เต็มรูปแบบค่อยต่อยอดภายหลัง


## Expected Output / Success Criteria

เมื่อจบ Day 19 ควรทำได้:

- อธิบายได้ว่า Bronze layer คือจุดรับ raw data พร้อม traceability
- อธิบายได้ว่าทำไม Bronze ไม่ควร clean/drop/deduplicate ข้อมูลหนักเกินไป
- สร้าง mock raw source files ใน Fabric Lakehouse Files ได้
- อ่าน raw CSV ด้วย explicit schema แบบ all-string เพื่อ preserve raw values ได้
- เพิ่ม audit columns เช่น `ingestion_timestamp`, `ingestion_date`, `source_file`, `source_system`, `batch_id`, `run_id` ได้
- เขียน Bronze Delta table ด้วย `append` mode ได้
- อ่าน Bronze table กลับมาตรวจ row count และ traceability ได้
- สร้าง simple ingestion log table เพื่อดู record count ต่อ batch/run ได้
- ทำ source-file level audit summary ได้
- เห็นชัดว่า invalid raw values ยังอยู่ใน Bronze เพื่อส่งต่อไปจัดการใน Silver/DQ days ถัดไป


## Final takeaway

Bronze ingestion ไม่ใช่แค่การอ่านไฟล์แล้ว write table แต่คือการทำให้ raw data กลายเป็นหลักฐานที่ query ได้และตามรอยได้

> Bronze ที่ดีควรตอบได้ว่า record นี้มาจากไหน, เข้าเมื่อไร, อยู่ batch/run ไหน และ source ส่งค่าเดิมมาอย่างไร

จำ 4 ข้อนี้ไว้ก่อน:

- preserve raw values ก่อน clean
- add audit columns ทุกครั้งที่ ingest
- append new batches อย่างระมัดระวัง
- validate row count และ traceability หลัง write เสมอ


## Optional cleanup

In [ ]:
# spark.sql(f"DROP TABLE IF EXISTS {bronze_table_name}")
# spark.sql(f"DROP TABLE IF EXISTS {bronze_log_table_name}")

# if os.path.exists(base_path):
#     shutil.rmtree(base_path)